In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA H100 80GB HBM3


In [ ]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/DatasetKorosi
!cat /content/drive/MyDrive/DatasetKorosi/data.yaml


data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid
path: /content/drive/MyDrive/DatasetKorosi

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['corrosion']


In [ ]:
%%bash
cat > yolov8n-hybrid.yaml << 'EOF'
# =====================================
# YOLOv8n Hybrid CNN–Transformer
# =====================================

nc: 1  # corrosion

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2fAttn, [512, 256, 8]]   # 🔥 FIXED
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024]]

head:
  - [-1, 1, Detect, [nc]]
EOF


In [ ]:
!sed -n '1,50p' yolov8n-hybrid.yaml


# =====================================
# YOLOv8n Hybrid CNN–Transformer
# =====================================

nc: 1  # corrosion

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2fAttn, [512, 256, 8]]   # 🔥 FIXED
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024]]

head:
  - [-1, 1, Detect, [nc]]


In [ ]:
!cat yolov8n-hybrid.yaml


# =====================================
# YOLOv8n Hybrid CNN–Transformer
# =====================================

nc: 1  # corrosion

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2fAttn, [512, 256, 8]]   # 🔥 FIXED
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024]]

head:
  - [-1, 1, Detect, [nc]]


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("Base YOLOv8 loaded ✅")


Base YOLOv8 loaded ✅


In [ ]:
model.train(
    data="/content/drive/MyDrive/DatasetKorosi/data.yaml",
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    project="/content/drive/MyDrive/CV_Corrosion_Project",
    name="Hybrid_CNN_Transformer",
    attn=True   # 🔥 INI KUNCI HYBRID
)


SyntaxError: '[31m[1mattn[0m' is not a valid YOLO argument. 

    Arguments received: ['yolo', '-f', '/root/.local/share/jupyter/runtime/kernel-0c52c303-a189-45a3-90e6-bff4d89469ed.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['segment', 'classify', 'obb', 'detect', 'pose']
                MODE (required) is one of ['track', 'predict', 'export', 'val', 'train', 'benchmark']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo11n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo11n-seg.pt source='https://youtu.be/LNwODJXcvt